# 5.3 반복적 코드 개선을 위한 RL

반복적 개선의 특징:
- **정의**: 오류 피드백을 받아 코드를 단계적으로 개선하는 과정
- **장점**: 한 번의 생성보다 여러 번의 개선이 더 높은 성공률 달성한다
- **성능**: 기본 42% → 3라운드 후 78% (약 86% 성능 향상)

**MDP 모델링**:
```
상태 s_t:
  - 현재 코드
  - 실행 오류 메시지
  - 실패한 테스트 케이스

행동 a_t:
  - 코드 수정 (LLM이 생성)

보상 r_t:
  - r_t = 현재라운드_테스트통과수 - 이전라운드_테스트통과수
  - 오류 감소 시 +1, 아니면 -0.5

전이:
  - s_{t+1} = exec(수정된 코드) → 새로운 오류 메시지
```

## 반복적 개선의 이점

### 왜 반복적 개선이 더 나은가?

**한 번의 생성 vs 반복적 개선**

```
한 번 생성 방식:
  정확성 = 기본 모델 성능 (예: 42%)
  → 낮은 성공률

반복적 개선 방식:
  라운드 1: 코드 생성 + 테스트 → 오류 메시지
  라운드 2: 오류 메시지 기반 수정 → 더 나은 코드
  라운드 3: 남은 오류 수정 → 최종 코드
  → 높은 성공률 (78%)
```

### 오류 분석과 반성(Reflection)

**MemoryStream을 통한 학습**:

```
라운드 1 오류:
  "ValueError: 범위 초과"
  → 메모리 저장: "범위 확인 로직 추가 필요하다"

라운드 2 수정:
  메모리 검색: "범위 관련 오류들..."
  피드백으로 범위 확인 로직 추가
  → 더 나은 수정

라운드 3 반성:
  지금까지의 모든 오류 분석
  "입력 검증 → 핵심 로직 → 출력" 순서 중요하다
  → 완전한 코드 재구성
```

In [1]:
from utils_openai import *
import numpy as np

heading("준비: 반복적 개선 RL")
print("✓ 오류 기반 피드백 메커니즘")
print("✓ 다단계 코드 개선")
print("✓ MemoryStream으로 패턴 학습")
print("✓ 보상 누적으로 최적화")

def strip_code_blocks(code_text):
    """LLM 응답에서 마크다운 코드 블록을 제거한다."""
    if "```" in code_text:
        code_text = code_text.replace("```python", "").replace("```", "").strip()
    return code_text


────────────────────────────────────────
  준비: 반복적 개선 RL
────────────────────────────────────────
✓ 오류 기반 피드백 메커니즘
✓ 다단계 코드 개선
✓ MemoryStream으로 패턴 학습
✓ 보상 누적으로 최적화


## 실습 1: 의도적 버그 있는 코드 → 자동 수정

버그 있는 코드에서 시작하여 자동으로 수정하는 루프를 구현한다.

In [2]:
heading("실습 1: 오류 기반 자동 수정")

buggy_code = """def find_median(numbers):
    # 버그: 정렬하지 않음
    n = len(numbers)
    if n % 2 == 1:
        return numbers[n // 2]
    else:
        return (numbers[n // 2 - 1] + numbers[n // 2]) / 2
"""

test_cases = [
    ([1, 2, 3], 2),
    ([4, 2, 1, 3], 2.5),
    ([10, 20, 30], 20)
]

step_print(1, "원본 코드", "의도적 버그 포함한다")
print(f"  {buggy_code}")

step_print(2, "라운드 1: 테스트", "버그 감지한다")

namespace = {}
failed_tests = []

exec(buggy_code, namespace)
find_median = namespace.get('find_median')

for nums, expected in test_cases:
    result = find_median(nums)
    if result != expected:
        failed_tests.append((nums, result, expected))
        print(f"  find_median({nums}) = {result}, 기대: {expected} [실패]")
    else:
        print(f"  find_median({nums}) = {result} [통과]")

if failed_tests:
    error_msg = "정렬되지 않은 배열에서 중앙값을 구하고 있다. 먼저 배열을 정렬해야 한다."
else:
    error_msg = "테스트 통과한다"

print(f"\n  오류 분석: {error_msg}")


────────────────────────────────────────
  실습 1: 오류 기반 자동 수정
────────────────────────────────────────
  [Step 1] 원본 코드
    의도적 버그 포함한다
  def find_median(numbers):
    # 버그: 정렬하지 않음
    n = len(numbers)
    if n % 2 == 1:
        return numbers[n // 2]
    else:
        return (numbers[n // 2 - 1] + numbers[n // 2]) / 2

  [Step 2] 라운드 1: 테스트
    버그 감지한다
  find_median([1, 2, 3]) = 2 [통과]
  find_median([4, 2, 1, 3]) = 1.5, 기대: 2.5 [실패]
  find_median([10, 20, 30]) = 20 [통과]

  오류 분석: 정렬되지 않은 배열에서 중앙값을 구하고 있다. 먼저 배열을 정렬해야 한다.


In [3]:
step_print(3, "라운드 2: 수정", "오류 피드백으로 코드 개선한다")

refinement_prompt = f"""다음 함수에 버그가 있다:
{buggy_code}

오류: {error_msg}

이 버그를 수정하여 올바른 중앙값을 반환하는 함수를 작성하라. 설명 없이 함수만 반환하라."""

refined_code = ask(refinement_prompt, system="파이썬 개발자이다. 정확한 함수만 작성한다.", temperature=0.3)
refined_code = strip_code_blocks(refined_code)

print(f"  수정된 코드:\n{refined_code}")

# 수정된 코드 테스트
namespace2 = {}
round2_passed = 0

exec(refined_code, namespace2)
find_median_v2 = namespace2.get('find_median')

print("\n  수정된 코드 테스트:")
if find_median_v2:
    for nums, expected in test_cases:
        result = find_median_v2(nums)
        if result == expected:
            round2_passed += 1
            print(f"  find_median({nums}) = {result} [통과]")
        else:
            print(f"  find_median({nums}) = {result}, 기대: {expected} [실패]")
else:
    print("  함수를 찾을 수 없다")

step_print(4, "보상 계산", "단계별 개선도 평가한다")
round1_passed = len(test_cases) - len(failed_tests)
print(f"  라운드 1 통과: {round1_passed}/{len(test_cases)}")
print(f"  라운드 2 통과: {round2_passed}/{len(test_cases)}")
print(f"  개선도: {round2_passed - round1_passed} 테스트 추가 통과")
print(f"  누적 보상: {round1_passed * 0.5 + round2_passed * 0.8:.2f}")

  [Step 3] 라운드 2: 수정
    오류 피드백으로 코드 개선한다
  수정된 코드:
def find_median(numbers):
    numbers.sort()
    n = len(numbers)
    if n % 2 == 1:
        return numbers[n // 2]
    else:
        return (numbers[n // 2 - 1] + numbers[n // 2]) / 2

  수정된 코드 테스트:
  find_median([1, 2, 3]) = 2 [통과]
  find_median([1, 2, 3, 4]) = 2.5 [통과]
  find_median([10, 20, 30]) = 20 [통과]
  [Step 4] 보상 계산
    단계별 개선도 평가한다
  라운드 1 통과: 2/3
  라운드 2 통과: 3/3
  개선도: 1 테스트 추가 통과
  누적 보상: 3.40


## 실습 2: MemoryStream을 활용한 패턴 학습

과거 오류 패턴을 저장하고 현재 오류 해결에 활용한다.

In [4]:
heading("실습 2: 메모리 기반 개선")

memory = MemoryStream()

step_print(1, "오류 패턴 저장", "과거 경험 메모리화한다")

error_patterns = [
    "IndexError 발생 시 배열 범위 확인한다",
    "TypeError 발생 시 입력 타입 변환한다",
    "정렬이 필요하면 먼저 sort() 호출한다",
    "빈 배열 처리는 if not array로 확인한다"
]

for pattern in error_patterns:
    memory.add(pattern, importance=0.8)
    print(f"  저장: {pattern}")

print(f"\n  총 {len(memory)} 개 패턴 저장")

step_print(2, "현재 오류 유사 패턴 검색", "메모리에서 관련 경험 검색한다")

current_error = "배열 중앙값을 구할 때 정렬 누락으로 잘못된 결과"
relevant = memory.retrieve(current_error, top_k=3)

print(f"  검색 쿼리: '{current_error}'")
print(f"\n  관련 경험 ({len(relevant)}개):")
for mem in relevant:
    print(f"    - {mem['content']}")

step_print(3, "지식 활용", "메모리 기반 수정한다")
print(f"  메모리의 '정렬이 필요하면 먼저 sort()' 패턴 적용한다")
print(f"  → 배열을 먼저 정렬한 후 중앙값 계산한다")


────────────────────────────────────────
  실습 2: 메모리 기반 개선
────────────────────────────────────────
  [Step 1] 오류 패턴 저장
    과거 경험 메모리화한다
  저장: IndexError 발생 시 배열 범위 확인한다
  저장: TypeError 발생 시 입력 타입 변환한다
  저장: 정렬이 필요하면 먼저 sort() 호출한다
  저장: 빈 배열 처리는 if not array로 확인한다

  총 4 개 패턴 저장
  [Step 2] 현재 오류 유사 패턴 검색
    메모리에서 관련 경험 검색한다
  검색 쿼리: '배열 중앙값을 구할 때 정렬 누락으로 잘못된 결과'

  관련 경험 (3개):
    - 빈 배열 처리는 if not array로 확인한다
    - IndexError 발생 시 배열 범위 확인한다
    - 정렬이 필요하면 먼저 sort() 호출한다
  [Step 3] 지식 활용
    메모리 기반 수정한다
  메모리의 '정렬이 필요하면 먼저 sort()' 패턴 적용한다
  → 배열을 먼저 정렬한 후 중앙값 계산한다


## 실습 3: 완전한 코드 개선 파이프라인

생성 → 실행 → 오류분석 → 수정 → 재실행의 전체 루프를 구현한다.

In [5]:
heading("실습 3: 전체 개선 파이프라인")

problem = "두 개의 정렬된 배열을 합쳐 정렬된 배열로 반환하는 함수 merge_sorted_arrays를 작성하라."
test_cases = [
    ([1, 3, 5], [2, 4, 6], [1, 2, 3, 4, 5, 6]),
    ([1, 2], [3, 4], [1, 2, 3, 4]),
    ([], [1, 2], [1, 2])
]

step_print(1, "문제", problem)
step_print(2, "테스트 케이스", f"{len(test_cases)}개")
for a, b, expected in test_cases:
    print(f"  merge({a}, {b}) = {expected}")

# 초기 생성
step_print(3, "라운드 1: 생성", "초기 코드 생성한다")
v1_code = ask(problem, system="설명 없이 파이썬 함수만 반환한다.", temperature=0.5)
v1_code = strip_code_blocks(v1_code)
print(f"  생성 완료: {len(v1_code)} 글자")

v1_passed = 0
v1_error = ""
namespace = {}
exec(v1_code, namespace)
merge_func = namespace.get('merge_sorted_arrays')

if merge_func:
    for a, b, expected in test_cases:
        if merge_func(a, b) == expected:
            v1_passed += 1
else:
    v1_error = "함수를 찾을 수 없다"

print(f"  결과: {v1_passed}/{len(test_cases)} 통과")
if v1_error:
    print(f"  오류: {v1_error}")

# 수정 라운드
step_print(4, "라운드 2: 수정", "피드백 기반 수정한다")
feedback1 = f"현재 {v1_passed}/{len(test_cases)} 테스트 통과. {v1_error if v1_error else '남은 오류 없음'}"
v2_code = ask(
    f"{problem}\n\n현재 코드:\n{v1_code}\n\n피드백: {feedback1}\n\n개선된 함수만 반환하라.",
    system="파이썬 함수를 개선한다. 설명 없이 함수만 반환한다.",
    temperature=0.3
)
v2_code = strip_code_blocks(v2_code)
print(f"  수정 완료")

v2_passed = 0
namespace = {}
exec(v2_code, namespace)
merge_func = namespace.get('merge_sorted_arrays')

if merge_func:
    for a, b, expected in test_cases:
        if merge_func(a, b) == expected:
            v2_passed += 1

print(f"  결과: {v2_passed}/{len(test_cases)} 통과")

step_print(5, "성능 비교", "라운드별 개선도")
print(f"  라운드 1: {v1_passed}/{len(test_cases)}")
print(f"  라운드 2: {v2_passed}/{len(test_cases)}")
print(f"  개선: {v2_passed - v1_passed} 테스트")
if v2_passed == len(test_cases):
    print(f"  ✓ 완전 성공")
else:
    print(f"  계속 개선 필요")


────────────────────────────────────────
  실습 3: 전체 개선 파이프라인
────────────────────────────────────────
  [Step 1] 문제
    두 개의 정렬된 배열을 합쳐 정렬된 배열로 반환하는 함수 merge_sorted_arrays를 작성하라.
  [Step 2] 테스트 케이스
    3개
  merge([1, 3, 5], [2, 4, 6]) = [1, 2, 3, 4, 5, 6]
  merge([1, 2], [3, 4]) = [1, 2, 3, 4]
  merge([], [1, 2]) = [1, 2]
  [Step 3] 라운드 1: 생성
    초기 코드 생성한다
  생성 완료: 352 글자
  결과: 3/3 통과
  [Step 4] 라운드 2: 수정
    피드백 기반 수정한다
  수정 완료
  결과: 3/3 통과
  [Step 5] 성능 비교
    라운드별 개선도
  라운드 1: 3/3
  라운드 2: 3/3
  개선: 0 테스트
  ✓ 완전 성공


## 요약: 반복적 코드 개선 RL

### 핵심 개념

**반복적 개선의 강점**:

1. **오류 기반 학습**
   - 각 라운드의 오류 메시지가 구체적 피드백 제공한다
   - 한 번의 생성보다 정확한 수정 가능하다

2. **누적 보상**
   - 라운드 t에서 r_t만큼 개선한다
   - 총 누적 보상 = Σ r_t (지수 감소)

3. **메모리 기반 최적화**
   - 과거 오류 패턴 저장한다
   - 현재 오류와 유사한 과거 해결책 활용한다

### MDP 분석

- **상태**: (코드, 오류메시지, 테스트결과)
- **행동**: LLM이 생성하는 수정 코드
- **보상**: 테스트 통과 증가 개수
- **에피소드 길이**: 최대 3-5 라운드 (시간 제약)

### 성능 특성

- **라운드 1**: 기본 모델 성능 (약 42%)
- **라운드 2**: 라운드 1 + 오류 수정 (약 60%)
- **라운드 3**: 더 정교한 수정 (약 78%)
- **라운드 4+**: 한계 도달 (수확 감소)